# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced by their `@id` for reproducibility and clarity.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object (not as a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id`.

We will list the record set `@id`s, and for each, the corresponding field `@id`s and column `@id`s.

In [ ]:
# List all record sets, fields, and columns by their `@id`

record_sets_info = []
for recordset in dataset.record_sets():
    recordset_id = recordset['@id']
    fields = recordset.get('field', [])
    # Ensure fields is always a list
    if isinstance(fields, dict):
        fields = [fields]
    # Each field can have a column
    field_infos = []
    for field in fields:
        field_id = field['@id']
        column = field.get('column', None)
        column_id = column['@id'] if column is not None else None
        field_infos.append({'field_id': field_id, 'column_id': column_id})
    record_sets_info.append({'recordset_id': recordset_id, 'fields': field_infos})

print('Record Sets Overview:')
for info in record_sets_info:
    print(f"Record Set @id: {info['recordset_id']}")
    for field in info['fields']:
        print(f"  Field @id: {field['field_id']}, Column @id: {field['column_id']}")

# Optionally, preview a sample record from each record set
for info in record_sets_info:
    rec_id = info['recordset_id']
    records = list(dataset.records(record_set=rec_id))
    if records:
        print(f"Sample record for {rec_id}: {records[0]}")


## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Extract data from each record set, referenced by their `@id`
# For this dataset, let's assemble the list of all available record set `@id`s

record_set_ids = [info['recordset_id'] for info in record_sets_info]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Choose the primary record set for EDA (typically the one with most survey columns)
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id and primary_record_set_id in dataframes:
    df = dataframes[primary_record_set_id]
    print(f"Fields/Columns in Record Set {primary_record_set_id}:")
    print(df.columns.tolist())
    df.head()
else:
    print('No record set dataframes found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as outlier removal, filtering, normalizing numeric fields, or grouping by key demographic attributes.

**All field references are via their `@id`.**

In [ ]:
# Explore numeric fields (such as GAD-7, PHQ-9, PSQ scores)
import numpy as np

# Find likely numeric field candidates by looking for typical mental health scores
numeric_field_ids = [col for col in df.columns if ('gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower())]
if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]
else:
    # fallback, pick first numeric-type column
    numeric_field_id = df.select_dtypes(include='number').columns[0]

# Filter outliers: e.g. only records where score > threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic field ('age', 'sex', 'level_of_education', etc.) using field @id
group_field_candidates = [col for col in df.columns if ('age' in col.lower() or 'education' in col.lower() or 'sex' in col.lower())]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print('No demographic field available for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields. We will show the distribution of a selected numeric score and how it relates to a demographic attribute, using column `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
plt.figure(figsize=(8, 6))
sns.histplot(df[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id} scores")
plt.xlabel(f"{numeric_field_id}")
plt.ylabel("Count")
plt.show()

# If available, visualize grouped means
if group_field_candidates:
    plt.figure(figsize=(10,6))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Average {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Average {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print('No suitable group field for grouped visualization.')

## 6. Conclusion
In this notebook, we demonstrated AI-ready FAIR data exploration with `mlcroissant`, referencing entities by their `@id`. We loaded survey data, reviewed schema structure, extracted records, filtered and normalized key scores, and visualized distributions.

- **Key fields (by `@id`) enable reproducible data selection and transformation.**
- **Exploring demographic correlations, such as education and age, supports public health research.**
- **This dataset exemplifies open, standardized, and AI-ready survey data for mental health analysis in Africa.**

You can reuse and adapt this notebook for further research, ensuring all schema and record elements are referenced by `@id` for modularity and clarity.